In [33]:
import pandas as pd
import numpy as np
from scipy.stats import spearmanr
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.cross_decomposition import PLSRegression

In [8]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [10]:
df_expr = pd.read_csv(r"/content/drive/MyDrive/Courses/ECE542/7194372/Input_data-2022-001/orthogroup_expression/zma_osa_sbi_ortho_hai_expr_all.tsv", sep='	')

In [11]:
print(df_expr)

              Gene.ID  mean_leaf.section.1  mean_leaf.section.2  \
0      ORTHO04M000001             4.546497             6.520382   
1      ORTHO04M000002             5.568790            10.509554   
2      ORTHO04M000003             5.755689             4.791018   
3      ORTHO04M000004             5.931737             7.356886   
4      ORTHO04M000005            22.267045            31.086364   
...               ...                  ...                  ...   
18444     zma-MIR399a             0.000000             0.000000   
18445     zma-MIR399f             0.000000             0.000000   
18446     zma-MIR408b             0.000000             0.000000   
18447     zma-MIR528a             0.000000             0.000000   
18448      zma-MIR529             0.000000             0.000000   

       mean_leaf.section.3  mean_leaf.section.4  mean_leaf.section.6  \
0                 6.253503             6.826752             6.745223   
1                11.924204             7.954140    

In [21]:
# load downloaded TF lists (lists from PlantDB)
# put them into one set of unique Transcription Factor Gene IDs
# Assuming the TF list files are tab-separated and the actual gene ID is in the 'Gene_ID' column
maize_tfs_df = pd.read_csv('/content/drive/MyDrive/Courses/ECE542/7194372/Zma_TF_list.txt', sep='\t', header=0)
rice_tfs_df = pd.read_csv('/content/drive/MyDrive/Courses/ECE542/7194372/Osj_TF_list.txt', sep='\t', header=0)
sorghum_tfs_df = pd.read_csv('/content/drive/MyDrive/Courses/ECE542/7194372/Sbi_TF_list.txt', sep='\t', header=0)

# Extracting the Gene_ID column
maize_tfs = maize_tfs_df['Gene_ID'].tolist()
rice_tfs = rice_tfs_df['Gene_ID'].tolist()
sorghum_tfs = sorghum_tfs_df['Gene_ID'].tolist()

all_tf_gene_ids = set(maize_tfs + rice_tfs + sorghum_tfs)

# use the mapping file to find which OGs these genes belong to
# 'orthos_maize_outward.tsv' contains both 'query_gene' and 'OG'
df_mapping = pd.read_csv('/content/drive/MyDrive/Courses/ECE542/7194372/Input_data-2022-001/orthogroups/maize_outward/orthos_maize_outward.tsv', sep='\t')

# find OGs where the orthologous_gene is in our TF list
tf_og_matches = df_mapping[df_mapping['orthologous_gene'].isin(all_tf_gene_ids)]['OG'].unique()

# convert 'OGxxx' to the 'ORTHO04Mxxxxxx' format used in your df_expr
# Based on your notebook: OG001 -> ORTHO04M000001
tf_ortho_ids = [f"ORTHO04M{og[2:].zfill(6)}" for og in tf_og_matches]

In [25]:
df_expr_indexed = df_expr.set_index('Gene.ID')

# Identify which rows (now indexed by Gene.ID) are TFs and which are Targets
tf_ids_in_matrix = df_expr_indexed.index.intersection(tf_ortho_ids).tolist()
target_ids_in_matrix = df_expr_indexed.index.difference(tf_ortho_ids).tolist()

# X = Predictors (Transcription Factors), Y = Responses (Target Genes)
# Transpose so rows are 'Experiments' and columns are 'Genes'
X = df_expr_indexed.loc[tf_ids_in_matrix].T
Y = df_expr_indexed.loc[target_ids_in_matrix].T

print(f"Predictor Matrix (TFs): {X.shape}")
print(f"Response Matrix (Targets): {Y.shape}")

Predictor Matrix (TFs): (698, 556)
Response Matrix (Targets): (698, 17893)


In [28]:
print(X)

Gene.ID              ORTHO04M000128  ORTHO04M000185  ORTHO04M000207  \
mean_leaf.section.1       16.600000        6.852941        3.590909   
mean_leaf.section.2        9.350000        6.917647        3.645455   
mean_leaf.section.3        5.510000        6.752941        4.154545   
mean_leaf.section.4        2.975000       12.529412        5.009091   
mean_leaf.section.6        3.070000        6.117647        4.309091   
...                             ...             ...             ...   
SRR9203000_Mean            8.346732        1.702465       19.362952   
SRR9203001_Mean            8.230350        3.398786       13.368596   
SRR9203002_Mean            5.482242        2.399818       13.711582   
SRR9203003_Mean            3.165239        1.283216       15.269380   
SRR9203005_Mean           11.182696        4.045289       20.045773   

Gene.ID              ORTHO04M000208  ORTHO04M000213  ORTHO04M000231  \
mean_leaf.section.1        4.281250        2.422222        3.873333   
mean_

In [32]:
# components = 3
pls = PLSRegression(n_components=3)

# Handle NaN values: PLSRegression does not accept missing values.
# Filling with 0 is a common strategy in gene expression for missing or non-expressed genes.
X = X.fillna(0)
Y = Y.fillna(0)

# Fit the model: Using the TF "Control Center" to predict the "Factory" output
pls.fit(X, Y)

# This gives you the 'estimated' expression data for target genes
Y_pred = pls.predict(X)

# Calculate how well the TFs predict each target gene (R-squared)
score = pls.score(X, Y)
print(f"Total variance explained: {score:.4f}")

Total variance explained: 0.2304


In [39]:
import seaborn as sns

# Ensure Y is a numpy array for consistent operations with Y_pred
Y_actual_np = Y.to_numpy()

# Initialize a list to store Spearman correlation coefficients for each target gene
spearman_correlations = []

# Iterate through each target gene (column) to calculate correlation
# Y_pred and Y_actual_np have shape (n_samples, n_features)
for i in range(Y_actual_np.shape[1]):
    # Calculate Spearman correlation for the i-th target gene
    # between its actual values and predicted values across all samples
    corr, _ = spearmanr(Y_actual_np[:, i], Y_pred[:, i])
    spearman_correlations.append(corr)

# Convert the list of correlations to a pandas Series for easier analysis
spearman_series = pd.Series(spearman_correlations, index=Y.columns)

# Display descriptive statistics of the correlations
print("Descriptive Statistics of Spearman Correlations for Target Genes:")
display(spearman_series.describe())

# Optionally, display the first few correlations
print("\nTop 10 Spearman Correlations:")
display(spearman_series.nlargest(10))

print("\nBottom 10 Spearman Correlations (can indicate negative correlation or poor prediction):")
display(spearman_series.nsmallest(10))

mean_rho = np.nanmean(spearman_series)
print(f"Mean Spearman Correlation: {mean_rho:.2f}")

Descriptive Statistics of Spearman Correlations for Target Genes:


,0
count,17893.000000
mean,0.473699
std,0.292592
min,-0.392400
25%,0.178542
50%,0.499115
75%,0.747398
max,0.962975



Top 10 Spearman Correlations:


,0
Gene.ID,
ORTHO04M000688,0.962975
ORTHO04M002110,0.959836
ORTHO04M003276,0.957315
ORTHO04M001953,0.952293
ORTHO04M001323,0.951650
ORTHO04M003472,0.951615
ORTHO04M004052,0.951573
ORTHO04M000592,0.951126
ORTHO04M006083,0.950831



Bottom 10 Spearman Correlations (can indicate negative correlation or poor prediction):


,0
Gene.ID,
ORTHO04M008117,-0.392400
ORTHO04M011036,-0.347512
ORTHO04M010514,-0.335474
ORTHO04M000289,-0.324260
ORTHO04M010880,-0.293288
ORTHO04M011091,-0.280535
ORTHO04M001722,-0.279296
ORTHO04M008985,-0.252235
ORTHO04M010302,-0.248333


Mean Spearman Correlation: 0.47
